# Whispers — Phase 1 GRPO training (Unsloth + TRL, Kaggle 1×T4)

This notebook runs end-to-end on a free Kaggle session with **a single T4 GPU**.
It is the *single-GPU* sibling of:
- `train_whispers_grpo_kaggle.ipynb` — multi-GPU (2×T4 sharded via `device_map='auto'` + Liger-Kernel)
- `train_whispers_grpo.ipynb` — free Colab T4

Pipeline:
1. Clones the Whispers OpenEnv repo and installs deps (Unsloth + TRL).
2. Loads `Qwen/Qwen2.5-1.5B-Instruct` via Unsloth (4-bit + LoRA r=16).
3. Drives a GRPO-style rollout that calls the in-process `WhispersEnv` for up to `max_steps` per episode.
4. Logs metrics to WandB and saves three plots into `assets/`.

### Kaggle setup
Kaggle's GPU options are *"GPU T4 ×2"* and *"GPU P100"*. Pick **GPU T4 ×2** (P100 is Pascal, sm_60, which is *not* supported by Unsloth's Triton kernels). This notebook then pins `CUDA_VISIBLE_DEVICES=0` so we only use the first T4 — Unsloth is single-GPU only.

If you'd rather use both T4s, switch to [`train_whispers_grpo_kaggle.ipynb`](train_whispers_grpo_kaggle.ipynb) which uses HF + Liger-Kernel + sharding instead.

**Theme**: Multi-Agent Interactions (cooperation, competition, negotiation, coalition formation).

> NOTE: All long-running cells are wrapped in `try/except` so a re-run without a GPU still produces the synthetic curves used by the headline plots.

## 0. Environment setup

Detects whether we're on Kaggle, Colab or local; clones the repo into the right place; pins to a single GPU even if Kaggle gave us two.

In [ ]:
import os, sys, subprocess

# Pin to a single GPU BEFORE any CUDA / torch / unsloth import.
# Unsloth is single-GPU only; if Kaggle gave us 2 T4s we only use the first.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

if 'kaggle_secrets' in sys.modules or os.path.exists('/kaggle/working'):
    PLATFORM = 'kaggle'
elif 'google.colab' in sys.modules:
    PLATFORM = 'colab'
else:
    PLATFORM = 'local'

REPO_URL = os.environ.get('WHISPERS_REPO', 'https://huggingface.co/spaces/varn03/whispers')
if PLATFORM == 'kaggle':
    REPO_DIR = '/kaggle/working/whispers'
elif PLATFORM == 'colab':
    REPO_DIR = '/content/whispers'
else:
    REPO_DIR = '..'

if PLATFORM in ('kaggle', 'colab') and not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

sys.path.insert(0, REPO_DIR)
print(f'platform={PLATFORM}  repo={REPO_DIR}  CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES")}')

In [ ]:
%%capture
# IMPORTANT: do NOT pin numpy / matplotlib / torch here.
# Kaggle pre-loads them into the kernel and pip-upgrading mid-session leads to
# `RuntimeError: numpy was upgraded mid-session` when Unsloth's C extensions
# get imported. We keep that part minimal and rely on Kaggle's defaults; the
# next cell auto-restarts the runtime if anything heavy still got bumped.
#
# Strategy for the HF stack:
#   * Let Unsloth choose its own transformers/tokenizers/huggingface-hub
#     versions (it pins them in its own setup.py to match the model classes
#     it patches at import time, e.g. Qwen3Attention which only exists in
#     transformers >=4.51). DON'T downgrade after the fact — that's what
#     caused `NameError: name 'Qwen3Attention' is not defined`.
#   * Just add TRL on top (Unsloth does not bundle it). TRL >=0.16 picks up
#     the new transformers _get_train_sampler(self, dataset) signature, so
#     the Unsloth compiled cache (regenerated each session, see safety-net
#     cell below) will use the matching signature.
if PLATFORM in ('kaggle', 'colab'):
    !pip install -q -U pip
    # Unsloth + its own consistent transformers/tokenizers/hub set.
    # The `[colab-new]` extras work for Kaggle too — same Linux + CUDA toolchain.
    !pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
    # Add TRL only (Unsloth doesn't ship it). >=0.16 matches the new
    # transformers signatures Unsloth git/main is patched against.
    !pip install -q --no-deps 'trl>=0.16,<0.20'
    !pip install -q wandb 'pydantic>=2.9' python-dotenv
else:
    print('Local mode — assumes deps already installed')

In [ ]:
# Safety net for two failure modes that bite every Kaggle/Colab Unsloth run:
#
# 1) ABI mismatch: pip just upgraded a native lib already loaded in the kernel
#    (numpy / torch). The in-memory C extensions no longer match the new .so
#    files Unsloth will dlopen, and the next import crashes hard. We detect
#    the version skew and force a one-time runtime restart. pip caches make
#    the second pass fast.
#
# 2) Stale Python-module cache for HF/TRL libs: if a previous run already
#    imported e.g. `transformers` or `trl`, and pip then upgraded them on
#    disk, `sys.modules['transformers']` is still pointing at the OLD module
#    object. Unsloth's compiled cache then references symbols that disagree
#    with what it patches (NameError: Qwen3Attention is the canonical
#    smoking gun). We restart in that case too.
#
# 3) Stale Unsloth compiled-cache dir on disk: this dir contains generated
#    .py shims that hardcode the EXACT method signatures from whatever
#    transformers/trl versions were installed when it was last regenerated.
#    If those have moved (every time you change pins), the next training
#    run blows up with cryptic TypeErrors. We ALWAYS wipe it — regenerating
#    is cheap (~30s on first model load).
if PLATFORM in ('kaggle', 'colab'):
    import importlib.metadata as _im
    import sys as _sys
    _restart = False

    import numpy as _np
    _np_disk = _im.version('numpy')
    if _np_disk != _np.__version__:
        print(f'numpy in kernel ({_np.__version__}) != on disk ({_np_disk}).')
        _restart = True
    else:
        print(f'numpy OK at {_np.__version__}')

    # Check every HF/Unsloth lib that the install cell touched. If the kernel
    # still holds a stale module object, force a restart so subsequent imports
    # see the on-disk version.
    for _pkg in ('trl', 'transformers', 'tokenizers',
                 'huggingface_hub', 'safetensors', 'peft', 'accelerate',
                 'unsloth', 'unsloth_zoo'):
        _pip_name = _pkg.replace('_', '-')
        if _pkg in _sys.modules:
            try:
                _disk_v = _im.version(_pip_name)
                _mem_v = getattr(_sys.modules[_pkg], '__version__', None)
                if _mem_v and _mem_v != _disk_v:
                    print(f'{_pkg} in kernel ({_mem_v}) != on disk ({_disk_v}).')
                    _restart = True
            except Exception:
                # Package was imported but pip can't find it — wipe the stale
                # module so re-import picks up the on-disk version cleanly.
                for _k in [k for k in _sys.modules if k.startswith(_pkg)]:
                    del _sys.modules[_k]
                print(f'cleared stale {_pkg} modules from sys.modules')

    # ALWAYS wipe Unsloth's compiled-cache directory. Whatever signature the
    # cache was generated against in a previous session is almost certainly
    # not the signature on disk now. Regenerating is automatic and fast.
    import shutil as _shutil
    for _cache in ('/kaggle/working/unsloth_compiled_cache',
                   '/content/unsloth_compiled_cache',
                   os.path.expanduser('~/.cache/unsloth_compiled_cache')):
        if os.path.isdir(_cache):
            try:
                _shutil.rmtree(_cache)
                print(f'cleared Unsloth compiled cache: {_cache}')
            except Exception as _e:
                print(f'could not remove {_cache}: {_e}')

    if _restart:
        print('Restarting runtime so the new ABI / module is picked up...')
        os.kill(os.getpid(), 9)
    else:
        print('all good — no restart needed.')

In [ ]:
# Credentials. We populate os.environ from (in order of preference):
#   1. anything already exported in the kernel,
#   2. Kaggle Secrets (sidebar Add-ons → Secrets) when running on Kaggle,
#   3. Colab Secrets (sidebar key icon) when running on Colab,
#   4. .env file in the repo root.
# Nothing is required — WANDB_API_KEY and HF_TOKEN are both optional. WANDB
# only enables cloud logging; HF_TOKEN is only needed for gated model pulls.
import os

def _maybe_set(key: str, value):
    if value and not os.environ.get(key):
        os.environ[key] = str(value)

if PLATFORM == 'kaggle':
    try:
        from kaggle_secrets import UserSecretsClient
        _ks = UserSecretsClient()
        for k in ('WANDB_API_KEY', 'HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
            try:
                _maybe_set(k, _ks.get_secret(k))
            except Exception:
                pass  # secret not added or not enabled for this notebook
    except ImportError:
        pass

if PLATFORM == 'colab':
    try:
        from google.colab import userdata
        for k in ('WANDB_API_KEY', 'HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
            try:
                _maybe_set(k, userdata.get(k))
            except Exception:
                pass
    except ImportError:
        pass

try:
    from dotenv import load_dotenv
    _candidate = os.path.join(REPO_DIR, '.env')
    if os.path.isfile(_candidate):
        load_dotenv(_candidate, override=False)
        print(f'Loaded credentials from {_candidate}')
except ImportError:
    pass

print('WANDB_API_KEY set?', bool(os.environ.get('WANDB_API_KEY')))
print('HF_TOKEN set?    ', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')))

In [ ]:
import os, json, math, random, time, subprocess, logging
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Quiet the Whispers env logger. Malformed-action `ToolError`s are an *expected*
# part of training (the agent is already penalised via shaping and the error
# is surfaced via info["tool_error"]), but they spam stderr.
# This works regardless of whether the in-process whispers/env.py emits the
# error at WARNING (older builds) or INFO (current builds) level.
logging.getLogger('whispers').setLevel(logging.ERROR)
logging.getLogger('whispers.env').setLevel(logging.ERROR)

ASSETS = Path(REPO_DIR) / 'assets'
ASSETS.mkdir(parents=True, exist_ok=True)
print('assets ->', ASSETS.resolve())

# GPU census — we expect exactly one T4 (compute capability 7.5) here.
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  cuda:{i}  {p.name}  cc={p.major}.{p.minor}  vram={p.total_memory/1e9:.1f}GB')
    if torch.cuda.device_count() > 1:
        print('Note: more than one GPU is visible. We pinned CUDA_VISIBLE_DEVICES=0 above '
              'so Unsloth only uses cuda:0. To use both T4s, switch to train_whispers_grpo_kaggle.ipynb.')
    cc = torch.cuda.get_device_properties(0)
    if cc.major < 7:
        print(f'WARNING: GPU compute capability {cc.major}.{cc.minor} (<7.0) is NOT supported by Unsloth. '
              'You are likely on Kaggle\'s P100 — switch to GPU T4 ×2 in Settings.')
else:
    print('No GPU detected — the model load will be skipped and the rest of the notebook will use synthetic curves.')

## 1. Smoke-test the environment
We confirm the `WhispersEnv` reset/step/grade contract before plugging it into GRPO.

In [ ]:
# Defensive sys.path setup — after a kernel restart (e.g. triggered by the
# safety-net cell), `sys.path` may not include the repo. We re-resolve it here
# so this cell works regardless of run order.
import sys as _sys, os as _os
for _c in ('/kaggle/working/whispers', '/content/whispers', '..'):
    if _os.path.isfile(_os.path.join(_c, 'whispers', 'env.py')):
        if _c not in _sys.path:
            _sys.path.insert(0, _c)
        break
else:
    raise FileNotFoundError(
        "Couldn't find the whispers/ package. Re-run the env-setup cell at the "
        "top of the notebook (it git-clones the repo into /kaggle/working/whispers)."
    )

from whispers.env import WhispersEnv
from whispers.models import WhispersAction

env = WhispersEnv(task_id='t1', seed=0)
obs = env.reset()
print('role=', obs.role, 'inbox=', len(obs.inbox), 'legal_tools=', obs.legal_tools)
obs, r, done, info = env.step(WhispersAction(tool='wait'))
print('step1 reward=', round(r.value, 3), 'done=', done)
report = {'location': {'value': 'Reactor 7', 'confidence': 0.5},
          'incident': {'value': 'fire alarm', 'confidence': 0.5},
          'time': {'value': '03:14', 'confidence': 0.5},
          'casualties': {'value': '0', 'confidence': 0.5}}
obs, r, done, info = env.step(WhispersAction(tool='publish', final_report=report))
print('terminal reward.value=', round(r.value, 3), 'episode_score=', round(info.get('episode_score', 0.0), 3))

## 2. Load Qwen2.5-1.5B-Instruct via Unsloth (LoRA r=16, 4-bit)

T4 is Turing (compute capability 7.5) so Unsloth's Triton kernels work natively. The 1.5B model in 4-bit uses ~3 GB VRAM, leaving ~12 GB for activations / KV-cache / LoRA grads, which is plenty for GRPO with `MAX_SEQ_LEN=2048`. Bump to `Qwen/Qwen2.5-3B-Instruct` if you want to use more VRAM.

In [ ]:
MODEL_NAME = os.environ.get('WHISPERS_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
MAX_SEQ_LEN = 2048
LORA_RANK = 16

model = tokenizer = None
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        dtype=None,                   # T4 → Unsloth picks fp16 automatically
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=LORA_RANK,
        lora_dropout=0.0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=3407,
    )
    # Quiet the upstream Qwen generation config: it ships with max_length=32768
    # which conflicts with our per-call max_new_tokens=128 and triggers a
    # warning on EVERY .generate() call (~21k calls during GRPO).
    model.generation_config.max_length = None
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    # Also dial down transformers' default verbosity so we only see real errors.
    import transformers
    transformers.utils.logging.set_verbosity_error()
    print('Unsloth model loaded OK')
    print('  model device:', next(model.parameters()).device)
    print('  trainable params: '
          f'{sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M / '
          f'{sum(p.numel() for p in model.parameters())/1e6:.0f}M')
except Exception as e:
    print(f'Skipping Unsloth load: {type(e).__name__}: {e}')
    print('Phase-1 training will be replaced with a synthetic curve so the rest of the notebook still runs.')
    print('Common causes:')
    print('  * GPU not enabled (Settings → Accelerator → GPU T4 ×2)')
    print('  * GPU is P100 (Pascal sm_60) — NOT supported by Unsloth, switch to GPU T4 ×2')
    print('  * numpy/torch ABI mismatch — Restart & Run All to re-trigger the safety-net cell')

## 3. Rollout helpers — connect env to GRPOTrainer

The trainable agent must emit a single JSON action per LLM call. We pass it through the same parser used by `inference.py` so behaviour at test time matches behaviour at train time.

In [ ]:
import importlib.util, sys as _sys
spec = importlib.util.spec_from_file_location('inference_root', f'{REPO_DIR}/inference.py')
inf = importlib.util.module_from_spec(spec)
_sys.modules['inference_root'] = inf
spec.loader.exec_module(inf)
SYSTEM_PROMPT = inf.SYSTEM_PROMPT
_build_user_prompt = inf._build_user_prompt
_coerce_action = inf._coerce_action
print('inference parser imported OK')

In [ ]:
TASK_MIX = ['t1', 't1', 't2', 't2', 't3', 't4', 't5']  # curriculum-weighted
MAX_STEPS_PER_EP = 18

def rollout_one(model, tokenizer, task_id: str, seed: int) -> dict:
    """Run one episode under the current model and return reward + bookkeeping."""
    env = WhispersEnv(task_id=task_id, seed=seed)
    obs = env.reset()
    rewards, actions = [], []
    cap = min(MAX_STEPS_PER_EP, obs.max_steps)
    for step in range(cap):
        prompt = SYSTEM_PROMPT + '\n\n' + _build_user_prompt(obs)
        if model is None:
            # Random scripted fallback so the loop still produces curves without a GPU
            tool = random.choice([t for t in obs.legal_tools if t != 'fact_check'])
            raw = json.dumps({'tool': tool,
                              'target_id': (obs.network_neighbors[0] if obs.network_neighbors else None),
                              'content': 'placeholder', 'confidence': 0.5,
                              'final_report': ({'location': {'value': 'Reactor 7', 'confidence': 0.4}}
                                               if tool == 'publish' else None)})
        else:
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            gen_kwargs = dict(
                max_new_tokens=128,
                do_sample=True,
                temperature=0.6,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
            out_ids = model.generate(**inputs, **gen_kwargs)
            raw = tokenizer.decode(out_ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        action, _err = _coerce_action(raw, obs)
        actions.append(action.tool)
        try:
            obs, r, done, info = env.step(action)
        except RuntimeError:
            break
        rewards.append(float(r.value))
        if done:
            break
    grade = env.grade_terminal()
    return {'task_id': task_id, 'seed': seed, 'rewards': rewards,
            'actions': actions, 'score': float(grade['value']),
            'breakdown': grade}

## 4. GRPO training loop

This cell runs the **real `trl.GRPOTrainer`** with `WhispersEnv` plugged in as the reward function. Each prompt drawn from `TASK_MIX` triggers `num_generations` parallel generations; each generation rolls out one episode and the episode's terminal `score` (in `[0,1]`) is returned as the reward. GRPO then computes group-relative advantages and takes one optimiser step.

Defaults below are tuned so a single T4 finishes in ~45 min. Bump `max_steps` if you have more wall-clock budget (Kaggle gives 9 h).

In [ ]:
USE_WANDB = bool(os.environ.get('WANDB_API_KEY'))
if USE_WANDB:
    import wandb
    wandb.login(key=os.environ['WANDB_API_KEY'])
    wandb.init(project='whispers-openenv',
               name='phase1-grpo-kaggle-1xt4',
               config={'tasks': TASK_MIX, 'lora_r': LORA_RANK,
                       'model': MODEL_NAME, 'platform': PLATFORM,
                       'gpu': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')})
else:
    print('WANDB_API_KEY not set; skipping cloud logging (curves still saved as PNGs).')

In [ ]:
# Real GRPOTrainer integration. We build a tiny torch.utils.data.Dataset of
# (task_id, seed, prompt) tuples, then GRPOTrainer pulls num_generations
# completions per row, calls our reward_fn (which actually rolls out an
# episode against WhispersEnv), and updates the LoRA weights.
#
# Falls back to a no-op rollout-only loop if Unsloth couldn't load.

history = {'step': [], 'task_id': [], 'score': [], 'cascade': [], 'cal': []}

# Try to import the real TRL GRPO API. If TRL is too old (<0.13) the import
# fails and we silently fall back to the rollout-only baseline loop below.
_HAVE_TRL_GRPO = False
try:
    from trl import GRPOConfig, GRPOTrainer  # noqa: F401  (added in trl 0.13.0)
    _HAVE_TRL_GRPO = True
except ImportError as _e:
    try:
        import trl as _trl
        _trl_ver = getattr(_trl, '__version__', '?')
    except ImportError:
        _trl_ver = 'not installed'
    print(f'trl is too old for GRPO (installed: {_trl_ver}, need >=0.13). '
          'Falling back to rollout-only baseline loop.')
    print('  Fix: re-run the install cell with `pip install -U "trl>=0.14,<0.16"` then Restart & Run All.')

if model is None or not _HAVE_TRL_GRPO:
    if model is None:
        print('No model loaded — running scripted-baseline rollouts only (no gradient update).')
    GRPO_STEPS = 300
    GENERATIONS_PER_STEP = 4
    t0 = time.time()
    for i in range(GRPO_STEPS):
        tid = TASK_MIX[i % len(TASK_MIX)]
        seed = 1000 + i
        scores, breakdowns = [], []
        for g in range(GENERATIONS_PER_STEP):
            out = rollout_one(model, tokenizer, task_id=tid, seed=seed + g)
            scores.append(out['score'])
            breakdowns.append(out['breakdown'])
        s = float(np.mean(scores))
        history['step'].append(i); history['task_id'].append(tid); history['score'].append(s)
        history['cascade'].append(float(np.mean([b['cascade_penalty'] for b in breakdowns])))
        history['cal'].append(float(np.mean([b['calibration'] for b in breakdowns])))
        if USE_WANDB:
            wandb.log({'grpo_step': i, f'score/{tid}': s, 'score/mean': s})
        if i % 25 == 0:
            print(f'step {i:4d}  task={tid}  score={s:.3f}  elapsed={(time.time()-t0)/60:.1f}m')
    print('done in', round((time.time()-t0)/60, 1), 'min')
else:
    from torch.utils.data import Dataset

    # Defensive monkey-patch: the Unsloth compiled-cache for GRPOTrainer may
    # define `_get_train_sampler(self)` (single-arg, pre-transformers-4.50)
    # while a too-new transformers calls it as `sampler_fn(dataset)` and crashes
    # with `TypeError: takes 1 positional argument but 2 were given`. We wrap
    # the cached method to absorb the extra positional `dataset` argument so
    # training proceeds even if the version pin in the install cell didn't take.
    try:
        import inspect
        _patched = 0
        for _name in ('UnslothGRPOTrainer', '_UnslothGRPOTrainer'):
            try:
                _mod_path = f'unsloth_compiled_cache.{_name}'
                _mod = __import__(_mod_path, fromlist=[_name])
                _cls = getattr(_mod, _name, None)
                if _cls is None:
                    continue
                _orig = _cls._get_train_sampler
                _params = list(inspect.signature(_orig).parameters)
                if len(_params) == 1:  # only `self`
                    def _wrapped(self, dataset=None, _orig=_orig):
                        return _orig(self)
                    _cls._get_train_sampler = _wrapped
                    _patched += 1
            except Exception:
                pass
        if _patched:
            print(f'patched _get_train_sampler signature on {_patched} Unsloth class(es)')
    except Exception:
        pass

    NUM_TRAIN_STEPS = 200          # ~45 min on a single T4
    NUM_GENERATIONS = 4
    DATASET_SIZE = NUM_TRAIN_STEPS * 2  # GRPO consumes one prompt per step (per_device bs=1)

    class WhispersPromptDataset(Dataset):
        """Each row is one (task_id, seed) pair rendered into the system+user prompt.
        We stash task_id/seed in extra columns so reward_fn can re-run the env."""
        def __init__(self, n):
            self.rows = []
            for i in range(n):
                tid = TASK_MIX[i % len(TASK_MIX)]
                seed = 1000 + i
                env_i = WhispersEnv(task_id=tid, seed=seed)
                obs = env_i.reset()
                prompt = SYSTEM_PROMPT + '\n\n' + _build_user_prompt(obs)
                self.rows.append({'prompt': prompt, 'task_id': tid, 'seed': seed})
        def __len__(self): return len(self.rows)
        def __getitem__(self, i): return self.rows[i]

    train_ds = WhispersPromptDataset(DATASET_SIZE)
    print(f'built training dataset: {len(train_ds)} prompts across {len(set(r["task_id"] for r in train_ds.rows))} task ids')

    _step_counter = {'i': 0}

    def reward_fn(prompts, completions, task_id=None, seed=None, **kw):
        """GRPO reward: roll out a full episode per generated completion.
        We only use the FIRST action emitted by the LLM (its raw completion)
        as the protagonist's first move; subsequent moves are sampled fresh
        from the (current, mid-update) model. This is a deliberate simplification
        — GRPO is stable under it because the credit assignment on the first-step
        return is dense enough."""
        scores = []
        for k, raw in enumerate(completions):
            tid = task_id[k] if isinstance(task_id, list) else task_id
            sd  = seed[k]    if isinstance(seed, list)    else seed
            env_k = WhispersEnv(task_id=tid, seed=sd)
            obs = env_k.reset()
            action, _err = _coerce_action(raw if isinstance(raw, str) else raw[0]['content'], obs)
            try:
                obs, _r, done, _info = env_k.step(action)
            except RuntimeError:
                done = True
            cap = min(MAX_STEPS_PER_EP - 1, obs.max_steps - 1)
            for _ in range(cap):
                if done: break
                p2 = SYSTEM_PROMPT + '\n\n' + _build_user_prompt(obs)
                inputs = tokenizer(p2, return_tensors='pt').to(model.device)
                out_ids = model.generate(
                    **inputs, max_new_tokens=128, do_sample=True, temperature=0.6, top_p=0.9,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                )
                raw2 = tokenizer.decode(out_ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
                act2, _ = _coerce_action(raw2, obs)
                try:
                    obs, _r, done, _info = env_k.step(act2)
                except RuntimeError:
                    break
            grade = env_k.grade_terminal()
            scores.append(float(grade['value']))
        s_mean = float(np.mean(scores))
        i = _step_counter['i']; _step_counter['i'] += 1
        tid_log = task_id[0] if isinstance(task_id, list) else task_id
        history['step'].append(i); history['task_id'].append(tid_log); history['score'].append(s_mean)
        history['cascade'].append(0.0); history['cal'].append(0.0)
        if USE_WANDB:
            wandb.log({'grpo_step': i, f'score/{tid_log}': s_mean, 'score/mean': s_mean})
        if i % 5 == 0:
            print(f'  reward batch {i:4d}  task={tid_log}  mean_score={s_mean:.3f}')
        return scores

    grpo_args = GRPOConfig(
        output_dir=str(Path(REPO_DIR) / 'ckpt' / 'phase1_t4'),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=NUM_GENERATIONS,
        max_prompt_length=1024,
        max_completion_length=128,
        learning_rate=5e-6,
        beta=0.04,                     # KL coefficient
        max_steps=NUM_TRAIN_STEPS,
        logging_steps=5,
        save_steps=100,
        bf16=False, fp16=True,         # T4 = Turing, fp16 only
        report_to='wandb' if USE_WANDB else 'none',
        remove_unused_columns=False,   # keep task_id/seed for reward_fn
    )
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[reward_fn],
        args=grpo_args,
        train_dataset=train_ds,
    )
    t0 = time.time()
    trainer.train()
    print('GRPO done in', round((time.time()-t0)/60, 1), 'min')

## 5. Save raw history (so plots cell can be re-run independently)

In [ ]:
import json as _json
(ASSETS / 'phase1_history.json').write_text(_json.dumps(history))
print('saved', ASSETS / 'phase1_history.json  (', len(history['step']), 'rows)')

## 6. Headline plots (committed PNGs)
Each plot has labelled axes + units + baseline reference lines.

In [ ]:
# Use an absolute path so it works from any cwd (Colab/Kaggle/local).
_plots_script = os.path.join(REPO_DIR, 'scripts', 'make_plots.py')
exec(compile(open(_plots_script).read(), _plots_script, 'exec'),
     {'__name__': '__main__', '__file__': _plots_script})

---
Phase 2 (self-play) lives in [`train_whispers_selfplay.ipynb`](train_whispers_selfplay.ipynb).

For multi-GPU (2×T4 on Kaggle) see [`train_whispers_grpo_kaggle.ipynb`](train_whispers_grpo_kaggle.ipynb).